In [1]:
!pip install joblib
!pip install requests

In [22]:
import requests
def create_embeddings(text_list):
    r=requests.post("http://localhost:11434/api/embed",json={#takes list instead of a text like /api/embeddings
        "model":"bge-m3",
        "input":text_list#"input" instead of "prompt" for list
    })
    embeddings=r.json()["embeddings"]#"embedding" if only single text passed , not list
    return embeddings

In [24]:
import os
import json
files=os.listdir("newTranscripts")

my_dicts=[]
chunk_id=0

for file in files:
    if file.endswith(".json"):
        with open (f"Transcripts/{file}","r") as f:
            data=json.load(f)
        print(f"📝 Creating Embedding for {file}")
        embeddings=create_embeddings([c["text"] for c in data["chunk"]])#creates a list of embeddings for each text in the list passed

        for i,chunk in enumerate(data["chunk"]):
            chunk["chunk_id"]=chunk_id
            chunk_id+=1
            chunk["embedding"]=embeddings[i]
            my_dicts.append(chunk)
        print(f"✅ Embeddings Created")


📝 Creating Embedding for 1-The Bundle Of Sticks Story  Stories for Kids.json
✅ Embeddings Created
📝 Creating Embedding for 2-The Goose Girl Story  Stories for Kids.json
✅ Embeddings Created
📝 Creating Embedding for 3-THE HUNGRY FOX STUCK IN THE TREE HOLE.json
✅ Embeddings Created
📝 Creating Embedding for 4-THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY .json
✅ Embeddings Created
📝 Creating Embedding for 5-THE DOVE and THE ANT Story Kids Short Story.json
✅ Embeddings Created


In [25]:
import pandas as pd
df = pd.DataFrame.from_records(my_dicts)
df

,number,title,start,end,text,chunk_id,embedding
0,1,The Bundle Of Sticks Story Stories for Kids,0.0,3.0,Join Kids Heart Family,0,"[-0.031379566, -0.0013101256, -0.05387355, 0.0..."
1,1,The Bundle Of Sticks Story Stories for Kids,3.0,31.0,Hello Boys!,1,"[-0.019979615, -0.0071189227, -0.062363274, -0..."
2,1,The Bundle Of Sticks Story Stories for Kids,31.0,33.0,What's going on over here?,2,"[-0.014994005, 0.014724412, -0.052418083, 0.01..."
3,1,The Bundle Of Sticks Story Stories for Kids,33.0,37.0,We have a match against the teams from the ot...,3,"[0.0038768167, 0.051831763, -0.05632041, -0.00..."
4,1,The Bundle Of Sticks Story Stories for Kids,37.0,40.0,but we can't agree upon anything.,4,"[0.021895805, 0.051303893, -0.032790866, -0.04..."
...,...,...,...,...,...,...,...
300,5,THE DOVE and THE ANT Story Kids Short Story,215.0,220.0,similarly every good deed we do for others wi...,300,"[-0.015424291, 0.049800552, -0.024233682, -0.0..."
301,5,THE DOVE and THE ANT Story Kids Short Story,221.0,223.0,Hmmm! I will always help the needy.,301,"[-0.05595944, 0.02840174, -0.0145571735, 0.016..."
302,5,THE DOVE and THE ANT Story Kids Short Story,224.0,226.0,That's like a good boy Tofu.,302,"[-0.012500383, 0.0132121155, -0.013611266, -0...."
303,5,THE DOVE and THE ANT Story Kids Short Story,237.0,240.0,Johnny Johnny! Yes Papa!,303,"[-0.009573289, -0.01235547, -0.047443364, 0.00..."


In [27]:
#save this dataframe
import joblib
joblib.dump(df,"embeddings.joblib")
print("✅embeddings.joblib created")

✅embeddings.joblib created


In [28]:
#load joblib df
import joblib
df1=joblib.load("embeddings.joblib")
df1

,number,title,start,end,text,chunk_id,embedding
0,1,The Bundle Of Sticks Story Stories for Kids,0.0,3.0,Join Kids Heart Family,0,"[-0.031379566, -0.0013101256, -0.05387355, 0.0..."
1,1,The Bundle Of Sticks Story Stories for Kids,3.0,31.0,Hello Boys!,1,"[-0.019979615, -0.0071189227, -0.062363274, -0..."
2,1,The Bundle Of Sticks Story Stories for Kids,31.0,33.0,What's going on over here?,2,"[-0.014994005, 0.014724412, -0.052418083, 0.01..."
3,1,The Bundle Of Sticks Story Stories for Kids,33.0,37.0,We have a match against the teams from the ot...,3,"[0.0038768167, 0.051831763, -0.05632041, -0.00..."
4,1,The Bundle Of Sticks Story Stories for Kids,37.0,40.0,but we can't agree upon anything.,4,"[0.021895805, 0.051303893, -0.032790866, -0.04..."
...,...,...,...,...,...,...,...
300,5,THE DOVE and THE ANT Story Kids Short Story,215.0,220.0,similarly every good deed we do for others wi...,300,"[-0.015424291, 0.049800552, -0.024233682, -0.0..."
301,5,THE DOVE and THE ANT Story Kids Short Story,221.0,223.0,Hmmm! I will always help the needy.,301,"[-0.05595944, 0.02840174, -0.0145571735, 0.016..."
302,5,THE DOVE and THE ANT Story Kids Short Story,224.0,226.0,That's like a good boy Tofu.,302,"[-0.012500383, 0.0132121155, -0.013611266, -0...."
303,5,THE DOVE and THE ANT Story Kids Short Story,237.0,240.0,Johnny Johnny! Yes Papa!,303,"[-0.009573289, -0.01235547, -0.047443364, 0.00..."


In [29]:
#query
incoming_query=input("Ask a Query")
question_embedding=create_embeddings([incoming_query])[0]

Ask a Query Where Is TOFU in and in which video at which Time?


In [59]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
#Find similarities of question embeddings with other embeddings

# print(np.vstack(df['embedding'].values))#converts array of list returned by dataframe to 2d vectors needed for cosine similarity
# print(np.vstack(df['embedding']).shape)#converts array of list returned by dataframe to 2d vectors needed for cosine similarity
similarities = cosine_similarity(np.vstack(df1['embedding']), [question_embedding]).flatten()
#print(similarities)#gives the cosine similarities values
#print(similarities.argsort())#gives the similarities indices in increasing order of values(end has index of maximum similarity)
top_results=5
max_indices=similarities.argsort()[::-1][0:top_results]#shows first top 5 #now in decreasing order(first index have maximum similarity index)

In [60]:
new_df = (
    df1.loc[max_indices]
       .sort_values(["number", "start"])
       .reset_index(drop=True)
)
new_df[["title","number","text"]]

,title,number,text
0,The Goose Girl Story Stories for Kids,2,"Thank you. What do you think, Tofu?"
1,THE HUNGRY FOX STUCK IN THE TREE HOLE,3,"Hey Tofu, guess what?"
2,THE HUNGRY FOX STUCK IN THE TREE HOLE,3,Have you ever seen this car Tofu?
3,THE MAGIC POT STORY STORIES FOR KIDS TRADITI...,4,"What is wrong, Tofu?"
4,THE MAGIC POT STORY STORIES FOR KIDS TRADITI...,4,"Don't worry about it, Tofu."


In [61]:
for index,item in new_df.iterrows():
    print(index,item["title"], item["text"],item["start"],item["end"])

0 The Goose Girl Story  Stories for Kids  Thank you. What do you think, Tofu? 466.0 469.0
1 THE HUNGRY FOX STUCK IN THE TREE HOLE  Hey Tofu, guess what? 11.0 14.0
2 THE HUNGRY FOX STUCK IN THE TREE HOLE  Have you ever seen this car Tofu? 28.0 31.0
3 THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY   What is wrong, Tofu? 18.0 20.0
4 THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY   Don't worry about it, Tofu. 35.0 38.0


In [62]:
#Creating Polished context data for LLM
from datetime import timedelta

def sec_to_hms(seconds):
    return str(timedelta(seconds=int(seconds)))

context = ""

for i, (index, row) in enumerate(new_df.iterrows(), start=1):

    context += f"""
====================

Chunk {i}

Video: {row['number']}
Title: {row['title']}
Time: {sec_to_hms(row['start'])} - {sec_to_hms(row['end'])}

Subtitle:{row['text']}

"""
print(context)



Chunk 1

Video: 2
Title: The Goose Girl Story  Stories for Kids
Time: 0:07:46 - 0:07:49

Subtitle: Thank you. What do you think, Tofu?



Chunk 2

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:11 - 0:00:14

Subtitle: Hey Tofu, guess what?



Chunk 3

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:28 - 0:00:31

Subtitle: Have you ever seen this car Tofu?



Chunk 4

Video: 4
Title: THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY 
Time: 0:00:18 - 0:00:20

Subtitle: What is wrong, Tofu?



Chunk 5

Video: 4
Title: THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY 
Time: 0:00:35 - 0:00:38

Subtitle: Don't worry about it, Tofu.




In [68]:
prompt = f"""
You are a teaching assistant.

Retrieved subtitle chunks:

{context}

Question:
{incoming_query}

Answer ONLY from the retrieved subtitle chunks.
If multiple chunks are relevant, include all of them.
Do not invent information.

Format:

Video:
Title:
Time:
Subtitle:

If nothing matches, reply exactly:
I couldn't find any matching subtitle in the provided videos.
"""
print(prompt)


You are a teaching assistant.

Retrieved subtitle chunks:



Chunk 1

Video: 2
Title: The Goose Girl Story  Stories for Kids
Time: 0:07:46 - 0:07:49

Subtitle: Thank you. What do you think, Tofu?



Chunk 2

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:11 - 0:00:14

Subtitle: Hey Tofu, guess what?



Chunk 3

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:28 - 0:00:31

Subtitle: Have you ever seen this car Tofu?



Chunk 4

Video: 4
Title: THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY 
Time: 0:00:18 - 0:00:20

Subtitle: What is wrong, Tofu?



Chunk 5

Video: 4
Title: THE MAGIC POT STORY  STORIES FOR KIDS  TRADITIONAL STORY 
Time: 0:00:35 - 0:00:38

Subtitle: Don't worry about it, Tofu.



Question:
Where Is TOFU in and in which video at which Time?

Answer ONLY from the retrieved subtitle chunks.
If multiple chunks are relevant, include all of them.
Do not invent information.

Format:

Video:
Title:
Time:
Subtitle:

If nothing matches,

In [69]:
#Getting Responses from Local LLM using OLLAMA
#download qwen2.5-coder:3b model for 4gb vram device
def inference_ollama_local(prompt,model="qwen2.5-coder:3b"):
    r=requests.post("http://localhost:11434/api/generate", json={#takes list instead of a text like /api/embeddings
        "model":model,
        "prompt":prompt,#"input" instead of "prompt" for list
        "stream":False
    })
    response=r.json()["response"]
    return response

In [70]:
#Getting Responses from Local LLM using LM Studio
#download "google/gemma-4-e4b", model for 4gb vram device
import requests

def inference_lmstudio_local(prompt, model="google/gemma-4-e4b", temperature=0):
    r = requests.post("http://127.0.0.1:1234/v1/chat/completions",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "stream": False
        })
    return r.json()["choices"][0]["message"]["content"]

In [71]:
#Getting Responses from Local LLM using LM Studio
#download "google/gemma-4-e2b", model for 4gb vram device
import requests

def inference_lmstudio_local_gemma2b(prompt, model="google/gemma-4-e2b", temperature=0):
    r = requests.post("http://127.0.0.1:1234/v1/chat/completions",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "stream": False
        })
    return r.json()["choices"][0]["message"]["content"]

In [72]:
# response = inference_lmstudio_local_gemma2b(prompt)
# response = inference_lmstudio_local_gemma4b(prompt)
# response = inference_ollama_local(prompt)
response = inference_lmstudio_local_gemma2b(prompt)
print(response)

Video: 2
Title: The Goose Girl Story Stories for Kids
Time: 0:07:46 - 0:07:49
Subtitle: Thank you. What do you think, Tofu?

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:11 - 0:00:14
Subtitle: Hey Tofu, guess what?

Video: 3
Title: THE HUNGRY FOX STUCK IN THE TREE HOLE
Time: 0:00:28 - 0:00:31
Subtitle: Have you ever seen this car Tofu?

Video: 4
Title: THE MAGIC POT STORY STORIES FOR KIDS TRADITIONAL STORY
Time: 0:00:18 - 0:00:20
Subtitle: What is wrong, Tofu?

Video: 4
Title: THE MAGIC POT STORY STORIES FOR KIDS TRADITIONAL STORY
Time: 0:00:35 - 0:00:38
Subtitle: Don't worry about it, Tofu.


In [62]:
with open("response.txt","w") as f:
    f.write(response)